In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.mart.fct_returns_flat AS
SELECT
  -- Grain identifier
  r.return_id,
  r.order_id,
  r.return_date,
  r.return_reason,
  r.refund_amount,

  -- Anomaly flag (mirrors the payment_date anomaly pattern)
  CASE WHEN r.return_date < o.order_date THEN TRUE ELSE FALSE END AS is_return_date_anomaly,

  -- Diagnostic: how long after the order was the return initiated
  DATEDIFF(r.return_date, o.order_date) AS days_to_return,

  -- Order context (for reconciliation and rate calculations)
  o.order_date,
  o.order_status,
  o.order_total,
  ROUND(r.refund_amount / NULLIF(o.order_total, 0), 4) AS refund_pct_of_order_total,

  -- Customer dimension
  c.customer_id,
  c.customer_segment,
  c.acquisition_channel,
  c.gender,
  c.city               AS customer_city,

  -- Customer home region
  cust_r.region_id      AS customer_region_id,
  cust_r.region_name    AS customer_region_name,
  cust_r.market_type    AS customer_market_type,

  -- Shipping / order region (for "are certain locations associated with high returns")
  o.region_id           AS shipping_region_id,
  ship_r.region_name    AS shipping_region_name,
  ship_r.market_type    AS shipping_market_type,

  -- Date dimension (based on return_date, since this table is return-centric)
  d.year,
  d.quarter,
  d.month,
  d.month_name,
  d.year_month,
  d.week,
  d.day_name,
  d.is_weekend,
  d.is_holiday,
  d.holiday_name

FROM ecommerce.clean.returns    r
JOIN ecommerce.clean.orders     o        ON r.order_id     = o.order_id
JOIN ecommerce.clean.customers  c        ON o.customer_id  = c.customer_id
LEFT JOIN ecommerce.base.regions cust_r  ON c.region_id     = cust_r.region_id
LEFT JOIN ecommerce.base.regions ship_r  ON o.region_id     = ship_r.region_id
LEFT JOIN ecommerce.base.date   d        ON r.return_date   = d.date;

### TEST 1 — Row count parity (mart should have exactly one row per return)

In [0]:
%sql
-- Diagnostic 1: How many duplicate dates, and how many copies of each?
SELECT date, COUNT(*) AS copies
FROM ecommerce.base.date
GROUP BY date
HAVING COUNT(*) > 1
ORDER BY copies DESC
LIMIT 20;

In [0]:
%sql
-- Diagnostic 2: Are the duplicates confined to the 2026 extension, or also in the original 2023-2025 range?
SELECT
  CASE WHEN date >= DATE '2026-01-01' THEN '2026 extension' ELSE 'original 2023-2025' END AS date_range,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT date) AS distinct_dates,
  COUNT(*) - COUNT(DISTINCT date) AS excess_rows
FROM ecommerce.base.date
GROUP BY CASE WHEN date >= DATE '2026-01-01' THEN '2026 extension' ELSE 'original 2023-2025' END;

In [0]:
%sql
-- Diagnostic 3: For one specific duplicated date, are the duplicate rows byte-identical,
-- or do they differ in some column (which would explain why DISTINCT didn't collapse them)?
SELECT *
FROM ecommerce.base.date
WHERE date = (
  SELECT date FROM ecommerce.base.date GROUP BY date HAVING COUNT(*) > 1 LIMIT 1
);

In [0]:
%sql
-- Expect: source_row_count = mart_row_count, row_diff = 0
SELECT
  (SELECT COUNT(*) FROM ecommerce.clean.returns) AS source_row_count,
  (SELECT COUNT(*) FROM ecommerce.mart.fct_returns_flat) AS mart_row_count,
  (SELECT COUNT(*) FROM ecommerce.clean.returns)
    - (SELECT COUNT(*) FROM ecommerce.mart.fct_returns_flat) AS row_diff;

### TEST 2 — Grain check (return_id must be unique)

In [0]:
%sql
-- Expect: 0 rows
SELECT return_id, COUNT(*) AS occurrences
FROM ecommerce.mart.fct_returns_flat
GROUP BY return_id
HAVING COUNT(*) > 1;

### TEST 3 — No fan-out risk from dimension tables (regions, date should each have unique keys)

In [0]:
%sql
-- Expect: 0 rows
SELECT 'regions' AS dim, region_id AS key_value, COUNT(*) AS dupes
FROM ecommerce.base.regions GROUP BY region_id HAVING COUNT(*) > 1
UNION ALL
SELECT 'date', date, COUNT(*)
FROM ecommerce.base.date GROUP BY date HAVING COUNT(*) > 1;

### TEST 4 — No orphaned rows dropped by inner joins

In [0]:
%sql
-- Expect: 0 rows
SELECT r.return_id
FROM ecommerce.clean.returns r
LEFT JOIN ecommerce.mart.fct_returns_flat f ON r.return_id = f.return_id
WHERE f.return_id IS NULL;

### TEST 5 — is_return_date_anomaly flag logic (should only be TRUE when return_date < order_date)

In [0]:
%sql
-- Expect: 0 rows
SELECT return_id, return_date, order_date, is_return_date_anomaly
FROM ecommerce.mart.fct_returns_flat
WHERE (return_date < order_date AND is_return_date_anomaly = FALSE)
   OR (return_date >= order_date AND is_return_date_anomaly = TRUE);

### TEST 6 — days_to_return should be consistent with is_return_date_anomaly (negative only when flagged)

In [0]:
%sql
-- Expect: 0 rows
SELECT return_id, days_to_return, is_return_date_anomaly
FROM ecommerce.mart.fct_returns_flat
WHERE (days_to_return < 0 AND is_return_date_anomaly = FALSE)
   OR (days_to_return >= 0 AND is_return_date_anomaly = TRUE);

### TEST 7 — refund_amount reconciliation (mart vs source)

In [0]:
%sql
-- Expect: refund_diff = 0
SELECT
  (SELECT ROUND(SUM(refund_amount), 2) FROM ecommerce.clean.returns) AS source_refunds,
  (SELECT ROUND(SUM(refund_amount), 2) FROM ecommerce.mart.fct_returns_flat) AS mart_refunds,
  (SELECT ROUND(SUM(refund_amount), 2) FROM ecommerce.clean.returns)
    - (SELECT ROUND(SUM(refund_amount), 2) FROM ecommerce.mart.fct_returns_flat) AS refund_diff;

### TEST 8 — refund_pct_of_order_total sanity (should not exceed 1.0 / 100%)

In [0]:
%sql
-- Expect: 0 rows. Any result means a refund exceeded the order's total -- a genuine
-- data-quality finding worth investigating, not necessarily a mart bug.
SELECT return_id, order_id, refund_amount, order_total, refund_pct_of_order_total
FROM ecommerce.mart.fct_returns_flat
WHERE refund_pct_of_order_total > 1.0;

### TEST 9 — Null profiling on key columns

In [0]:
%sql
-- Expect: 0 across all columns listed here
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(return_reason)        AS null_return_reason,
  COUNT(*) - COUNT(refund_amount)        AS null_refund_amount,
  COUNT(*) - COUNT(order_total)          AS null_order_total,
  COUNT(*) - COUNT(customer_segment)     AS null_segment,
  COUNT(*) - COUNT(year)                 AS null_date_join
FROM ecommerce.mart.fct_returns_flat;

### TEST 10 — Referential sanity (order_id and customer_id resolve to clean dimensions)

In [0]:
%sql
-- Expect: 0 rows for each check_type
SELECT 'order_id' AS check_type, f.order_id AS bad_key
FROM ecommerce.mart.fct_returns_flat f
LEFT JOIN ecommerce.clean.orders o ON f.order_id = o.order_id
WHERE o.order_id IS NULL
UNION ALL
SELECT 'customer_id', f.customer_id
FROM ecommerce.mart.fct_returns_flat f
LEFT JOIN ecommerce.clean.customers c ON f.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

### TEST 11 — Value range sanity (refund_amount should never be negative or zero)

In [0]:
%sql
-- Expect: 0 rows
SELECT *
FROM ecommerce.mart.fct_returns_flat
WHERE refund_amount <= 0;

### TEST 12 — Date join sanity (return_date should match the joined date-dimension row exactly)

In [0]:
%sql
-- Expect: 0 rows (date dimension was extended through 2026-03-31 to cover max return_date of 2026-01-29)
SELECT f.return_id, f.return_date, d.date AS joined_date
FROM ecommerce.mart.fct_returns_flat f
LEFT JOIN ecommerce.base.date d ON f.return_date = d.date
WHERE d.date IS NULL OR f.return_date != d.date;

### TEST 13 — Excluded fields truly absent (product_id/category_id should NOT be in this table — known scope limitation, see Q12 notes)
Run manually and confirm it errors with "column does not exist" — an error here is a PASS, not a failure.

In [0]:
%sql
-- Expect: query FAILS (column not found) — that failure is the pass condition
SELECT product_id, category_id, order_item_id FROM ecommerce.mart.fct_returns_flat LIMIT 1;

### TEST 14 — Distinct value spot-check on return_reason

In [0]:
%sql
-- Expect: only 'DAMAGED', 'WRONG ITEM', 'CUSTOMER CHANGED MIND', 'POOR QUALITY',
-- 'SIZE ISSUE', 'LATE DELIVERY' (UPPER-cased per the cleaning step)
SELECT DISTINCT return_reason FROM ecommerce.mart.fct_returns_flat;

### PROFILING (not pass/fail) — single-item vs multi-item order coverage for Q12
Documents what share of returns can be attributed to an exact product (single-item orders) vs. only estimated (multi-item orders). Record this % in the analysis write-up whenever the mart is rebuilt, since it determines how strongly Q12 can be answered.

In [0]:
%sql
WITH item_counts AS (
  SELECT order_id, COUNT(*) AS item_count
  FROM ecommerce.clean.order_items
  GROUP BY order_id
)
SELECT
  CASE WHEN ic.item_count = 1 THEN 'Single-item order' ELSE 'Multi-item order' END AS order_type,
  COUNT(DISTINCT r.return_id)   AS return_count,
  ROUND(100.0 * COUNT(DISTINCT r.return_id) / SUM(COUNT(DISTINCT r.return_id)) OVER (), 2) AS pct_of_returns
FROM ecommerce.mart.fct_returns_flat r
JOIN item_counts ic ON r.order_id = ic.order_id
GROUP BY CASE WHEN ic.item_count = 1 THEN 'Single-item order' ELSE 'Multi-item order' END;